In [3]:
from pathlib import Path
from collections import Counter
from itertools import combinations
import re
import shutil
import unicodedata

import pandas as pd
import numpy as np
import networkx as nx
import plotly.express as px
import plotly.graph_objects as go


# ---------------------------------------------------------
# CONFIGURACIÓN
# ---------------------------------------------------------

# Los CSV y matrices incluyen todos los términos encontrados.
# Las visualizaciones se limitan para conservar legibilidad.
TOP_FREQUENCY = 30
TOP_HEATMAP = 30
TOP_NETWORK = 60


def encontrar_raiz():
    actual = Path.cwd().resolve()

    for carpeta in (actual, *actual.parents):
        if (carpeta / "data" / "scopus.csv.gz").is_file():
            return carpeta

        candidata = carpeta / "PRE_13_scopus"
        if (candidata / "data" / "scopus.csv.gz").is_file():
            return candidata

    raise FileNotFoundError(
        "No se encontró data/scopus.csv.gz. "
        "El archivo debe existir para generar resultados reales."
    )


def normalizar_nombre(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]", "", texto.lower())


def buscar_columna(df, opciones):
    disponibles = {
        normalizar_nombre(columna): columna
        for columna in df.columns
    }

    for opcion in opciones:
        encontrada = disponibles.get(normalizar_nombre(opcion))
        if encontrada is not None:
            return encontrada

    return None


def separar_terminos(valor):
    """Scopus normalmente separa autores, keywords y afiliaciones con ;."""
    if pd.isna(valor):
        return []

    return [
        parte.strip()
        for parte in str(valor).split(";")
        if parte.strip()
    ]


def unicos(terminos):
    """Cada término cuenta una sola vez por documento."""
    return sorted(set(terminos))


def tabla_frecuencia(listas, nombre):
    contador = Counter(
        termino
        for documento in listas
        for termino in set(documento)
    )

    tabla = pd.DataFrame(
        contador.items(),
        columns=[nombre, "document_count"],
    )

    return tabla.sort_values(
        ["document_count", nombre],
        ascending=[False, True],
    ).reset_index(drop=True)


def matriz_coocurrencia(listas, etiquetas):
    """
    Diagonal: documentos donde aparece el término.
    Fuera de la diagonal: documentos donde aparecen ambos términos.
    """
    posiciones = {
        etiqueta: indice
        for indice, etiqueta in enumerate(etiquetas)
    }

    matriz = np.zeros(
        (len(etiquetas), len(etiquetas)),
        dtype=np.int64,
    )

    for documento in listas:
        indices = sorted({
            posiciones[termino]
            for termino in documento
            if termino in posiciones
        })

        for indice in indices:
            matriz[indice, indice] += 1

        for izquierda, derecha in combinations(indices, 2):
            matriz[izquierda, derecha] += 1
            matriz[derecha, izquierda] += 1

    return pd.DataFrame(
        matriz,
        index=etiquetas,
        columns=etiquetas,
    )


def crear_grafo(matriz):
    grafo = nx.Graph()

    for termino in matriz.index:
        grafo.add_node(
            termino,
            frequency=int(matriz.loc[termino, termino]),
        )

    valores = matriz.to_numpy()

    for i, j in zip(*np.where(np.triu(valores, k=1) > 0)):
        grafo.add_edge(
            matriz.index[i],
            matriz.columns[j],
            weight=int(valores[i, j]),
        )

    return grafo


def obtener_clusters(grafo):
    if grafo.number_of_nodes() == 0:
        return []

    if grafo.number_of_edges() == 0:
        return [{nodo} for nodo in sorted(grafo.nodes())]

    comunidades = nx.community.greedy_modularity_communities(
        grafo,
        weight="weight",
    )

    return sorted(
        comunidades,
        key=lambda comunidad: (-len(comunidad), sorted(comunidad)),
    )


def guardar_clusters(comunidades, ruta):
    lineas = [
        "Método: comunidades por modularidad greedy ponderada.",
        "Peso: cantidad de documentos compartidos.",
        "",
    ]

    if not comunidades:
        lineas.append("No se encontraron términos para formar comunidades.")

    for numero, comunidad in enumerate(comunidades, start=1):
        lineas.append(f"Cluster {numero} ({len(comunidad)} términos)")
        lineas.extend(f"  - {termino}" for termino in sorted(comunidad))
        lineas.append("")

    ruta.write_text("\n".join(lineas), encoding="utf-8")


def figura_vacia(titulo, mensaje="No hay datos disponibles."):
    figura = go.Figure()

    figura.add_annotation(
        text=mensaje,
        x=0.5,
        y=0.5,
        xref="paper",
        yref="paper",
        showarrow=False,
    )

    figura.update_layout(title=titulo)
    return figura


def guardar_html(figura, ruta):
    # Incluye Plotly dentro del HTML para abrirlo sin internet.
    figura.write_html(
        str(ruta),
        include_plotlyjs=True,
        full_html=True,
        config={
            "scrollZoom": True,
            "displaylogo": False,
            "responsive": True,
        },
    )


def dibujar_red(grafo, comunidades, titulo):
    if grafo.number_of_nodes() == 0:
        return figura_vacia(titulo)

    principales = sorted(
        grafo.nodes(),
        key=lambda nodo: (-grafo.nodes[nodo]["frequency"], nodo),
    )[:TOP_NETWORK]

    subgrafo = grafo.subgraph(principales).copy()

    posiciones = nx.spring_layout(
        subgrafo,
        seed=42,
        weight="weight",
    )

    cluster_por_nodo = {
        nodo: numero
        for numero, comunidad in enumerate(comunidades, start=1)
        for nodo in comunidad
    }

    aristas_x = []
    aristas_y = []

    for origen, destino in subgrafo.edges():
        x0, y0 = posiciones[origen]
        x1, y1 = posiciones[destino]

        aristas_x.extend([x0, x1, None])
        aristas_y.extend([y0, y1, None])

    figura = go.Figure()

    figura.add_trace(go.Scatter(
        x=aristas_x,
        y=aristas_y,
        mode="lines",
        line=dict(width=0.8, color="#b8c2cc"),
        hoverinfo="skip",
        showlegend=False,
    ))

    nodos = list(subgrafo.nodes())
    frecuencias = [
        subgrafo.nodes[nodo]["frequency"]
        for nodo in nodos
    ]
    maximo = max(frecuencias, default=1)

    figura.add_trace(go.Scatter(
        x=[posiciones[nodo][0] for nodo in nodos],
        y=[posiciones[nodo][1] for nodo in nodos],
        mode="markers+text",
        text=nodos,
        textposition="top center",
        customdata=[
            [subgrafo.nodes[nodo]["frequency"], cluster_por_nodo[nodo]]
            for nodo in nodos
        ],
        hovertemplate=(
            "%{text}<br>"
            "Documentos: %{customdata[0]}<br>"
            "Cluster: %{customdata[1]}"
            "<extra></extra>"
        ),
        marker=dict(
            size=[
                10 + 25 * np.sqrt(frecuencia / maximo)
                for frecuencia in frecuencias
            ],
            color=[cluster_por_nodo[nodo] for nodo in nodos],
            colorscale="Turbo",
            line=dict(width=1, color="white"),
            showscale=False,
        ),
        showlegend=False,
    ))

    figura.update_layout(
        title=f"{titulo} · hasta {TOP_NETWORK} términos principales",
        template="plotly_white",
        height=750,
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        margin=dict(l=40, r=40, t=90, b=40),
    )

    return figura


# ---------------------------------------------------------
# PROCESAMIENTO PRINCIPAL
# ---------------------------------------------------------

def main():
    raiz = encontrar_raiz()
    entrada = raiz / "data" / "scopus.csv.gz"
    salida = raiz / "submission"
    salida.mkdir(parents=True, exist_ok=True)

    # Lee el archivo comprimido sin descomprimirlo manualmente.
    # Detecta el delimitador a partir del contenido.
    df = pd.read_csv(
        entrada,
        compression="gzip",
        encoding="utf-8-sig",
        sep=None,
        engine="python",
    )

    df.columns = df.columns.str.strip()

    print(f"Documentos leídos: {len(df)}")
    print("Columnas disponibles:", list(df.columns))

    columna_autores = buscar_columna(
        df, ["Authors", "Author full names", "Author(s)"]
    )
    columna_keywords = buscar_columna(
        df, ["Author Keywords", "Keywords", "Index Keywords"]
    )
    columna_paises = buscar_columna(
        df, ["Countries", "Country", "Affiliation Countries"]
    )
    columna_afiliaciones = buscar_columna(
        df, ["Affiliations", "Authors with affiliations"]
    )
    columna_fuentes = buscar_columna(
        df, ["Source title", "Source", "Journal"]
    )
    columna_anio = buscar_columna(
        df, ["Year", "Publication Year"]
    )

    # No generar resultados vacíos silenciosamente por columnas desconocidas.
    faltantes = []

    for nombre, columna in [
        ("autores", columna_autores),
        ("palabras clave", columna_keywords),
        ("fuente", columna_fuentes),
        ("año", columna_anio),
    ]:
        if columna is None:
            faltantes.append(nombre)

    if columna_paises is None and columna_afiliaciones is None:
        faltantes.append("países o afiliaciones")

    if faltantes:
        raise ValueError(
            f"No se identificaron columnas de: {', '.join(faltantes)}. "
            "Revisa las columnas impresas y agrega sus nombres "
            "en las llamadas a buscar_columna()."
        )

    # Autores: se conservan los nombres del archivo.
    autores = df[columna_autores].apply(
        lambda valor: unicos(separar_terminos(valor))
    ).tolist()

    # Keywords: se usa una columna, con preferencia por Author Keywords.
    keywords = df[columna_keywords].apply(
        lambda valor: unicos([
            re.sub(r"\s+", " ", termino).casefold()
            for termino in separar_terminos(valor)
        ])
    ).tolist()

    alias_paises = {
        "usa": "United States",
        "u.s.a.": "United States",
        "united states of america": "United States",
        "uk": "United Kingdom",
        "england": "United Kingdom",
        "scotland": "United Kingdom",
        "wales": "United Kingdom",
        "northern ireland": "United Kingdom",
        "russian federation": "Russia",
        "republic of korea": "South Korea",
        "korea, republic of": "South Korea",
        "viet nam": "Vietnam",
    }

    def limpiar_pais(pais):
        pais = re.sub(r"\s+", " ", pais).strip()
        return alias_paises.get(pais.casefold(), pais)

    if columna_paises is not None:
        paises = df[columna_paises].apply(
            lambda valor: unicos([
                limpiar_pais(pais)
                for pais in separar_terminos(valor)
            ])
        ).tolist()

        print(f"Países obtenidos de: {columna_paises}")

    else:
        def extraer_paises(valor):
            encontrados = []

            for afiliacion in separar_terminos(valor):
                partes = [
                    parte.strip()
                    for parte in afiliacion.split(",")
                    if parte.strip()
                ]

                # Una afiliación sin comas no permite inferir el país.
                if len(partes) >= 2:
                    encontrados.append(limpiar_pais(partes[-1]))

            return unicos(encontrados)

        paises = df[columna_afiliaciones].apply(
            extraer_paises
        ).tolist()

        print(
            "Países inferidos del último componente de cada afiliación. "
            "Revisa country_frequency.csv para detectar nombres incorrectos."
        )

    fuentes = df[columna_fuentes].apply(
        lambda valor: [] if pd.isna(valor) or not str(valor).strip()
        else [str(valor).strip()]
    ).tolist()

    # 1–4. Frecuencias.
    authors_frequency = tabla_frecuencia(autores, "author")
    country_frequency = tabla_frecuencia(paises, "country")
    keywords_frequency = tabla_frecuencia(keywords, "keyword")
    source_frequency = tabla_frecuencia(fuentes, "source")

    tablas = {
        "authors_frequency.csv": authors_frequency,
        "country_frequency.csv": country_frequency,
        "keywords_frequency.csv": keywords_frequency,
        "source_frequency.csv": source_frequency,
    }

    for nombre, tabla in tablas.items():
        tabla.to_csv(salida / nombre, index=False)

    # 5–6. Matrices completas de coocurrencia.
    country_matrix = matriz_coocurrencia(
        paises,
        country_frequency["country"].tolist(),
    )
    keywords_matrix = matriz_coocurrencia(
        keywords,
        keywords_frequency["keyword"].tolist(),
    )

    country_matrix.to_csv(
        salida / "country_cooc_matrix.csv",
        index_label="country",
    )
    keywords_matrix.to_csv(
        salida / "keywords_cooc_matrix.csv",
        index_label="keyword",
    )

    # 7–8. Clusters calculados sobre las redes completas.
    country_graph = crear_grafo(country_matrix)
    keywords_graph = crear_grafo(keywords_matrix)

    country_clusters = obtener_clusters(country_graph)
    keywords_clusters = obtener_clusters(keywords_graph)

    guardar_clusters(
        country_clusters,
        salida / "country_clusters.txt",
    )
    guardar_clusters(
        keywords_clusters,
        salida / "keywords_clusters.txt",
    )

    # 9–10. Redes interactivas.
    guardar_html(
        dibujar_red(
            country_graph,
            country_clusters,
            "Colaboración entre países",
        ),
        salida / "country_collab_network.html",
    )

    guardar_html(
        dibujar_red(
            keywords_graph,
            keywords_clusters,
            "Coocurrencia de palabras clave",
        ),
        salida / "keywords_cooc_network.html",
    )

    # 11. Frecuencia de países.
    if country_frequency.empty:
        figura = figura_vacia("Documentos por país")
    else:
        top = country_frequency.head(TOP_FREQUENCY).iloc[::-1]

        figura = px.bar(
            top,
            x="document_count",
            y="country",
            orientation="h",
            title=f"Documentos por país · primeros {TOP_FREQUENCY}",
            labels={
                "document_count": "Documentos",
                "country": "País",
            },
            template="plotly_white",
        )

        figura.update_layout(height=max(450, 25 * len(top)))

    guardar_html(
        figura,
        salida / "country_frequency_plot.html",
    )

    # 12. Mapa mundial.
    if country_frequency.empty:
        figura = figura_vacia("Distribución mundial de documentos")
    else:
        figura = px.choropleth(
            country_frequency,
            locations="country",
            locationmode="country names",
            color="document_count",
            hover_name="country",
            color_continuous_scale="Blues",
            title="Distribución mundial de documentos",
            labels={"document_count": "Documentos"},
            projection="natural earth",
        )

    guardar_html(figura, salida / "world_map.html")

    # 13. Mapa de calor.
    if country_matrix.empty:
        figura = figura_vacia("Coocurrencia entre países")
    else:
        principales = country_frequency["country"].head(
            TOP_HEATMAP
        ).tolist()

        seleccion = country_matrix.loc[
            principales, principales
        ]

        figura = go.Figure(go.Heatmap(
            z=seleccion.to_numpy(),
            x=seleccion.columns.tolist(),
            y=seleccion.index.tolist(),
            colorscale="Blues",
            colorbar=dict(title="Documentos"),
            hovertemplate=(
                "%{y} / %{x}<br>"
                "Documentos: %{z}<extra></extra>"
            ),
        ))

        figura.update_layout(
            title=(
                "Coocurrencia entre países"
                "<br><sup>Diagonal: documentos por país; "
                "resto: documentos compartidos</sup>"
            ),
            height=850,
            margin=dict(l=180, r=60, t=110, b=180),
            xaxis=dict(tickangle=-45, automargin=True),
            yaxis=dict(automargin=True, autorange="reversed"),
        )

    guardar_html(
        figura,
        salida / "country_cooc_heatmap.html",
    )

    # 14. Documentos por año.
    anios = pd.to_numeric(df[columna_anio], errors="coerce")
    anios = anios[
        anios.notna() & anios.mod(1).eq(0)
    ].astype(int)

    documentos_anuales = (
        anios.value_counts()
        .sort_index()
        .rename_axis("year")
        .reset_index(name="document_count")
    )

    if documentos_anuales.empty:
        figura = figura_vacia("Documentos por año")
    else:
        figura = px.bar(
            documentos_anuales,
            x="year",
            y="document_count",
            title="Documentos por año",
            labels={
                "year": "Año",
                "document_count": "Documentos",
            },
            template="plotly_white",
        )

        figura.update_xaxes(tickformat="d")

    guardar_html(
        figura,
        salida / "documents_by_year.html",
    )

    # 15. Copia exacta del archivo de entrada comprimido.
    shutil.copy2(entrada, salida / "scopus.csv.gz")

    archivos_esperados = [
        "authors_frequency.csv",
        "country_clusters.txt",
        "country_collab_network.html",
        "country_cooc_heatmap.html",
        "country_cooc_matrix.csv",
        "country_frequency.csv",
        "country_frequency_plot.html",
        "documents_by_year.html",
        "keywords_clusters.txt",
        "keywords_cooc_matrix.csv",
        "keywords_cooc_network.html",
        "keywords_frequency.csv",
        "scopus.csv.gz",
        "source_frequency.csv",
        "world_map.html",
    ]

    faltantes = [
        nombre
        for nombre in archivos_esperados
        if not (salida / nombre).is_file()
    ]

    if faltantes:
        raise RuntimeError(f"No se generaron estos archivos: {faltantes}")

    print(f"\nSe generaron los {len(archivos_esperados)} archivos.")
    print(f"Columna de autores: {columna_autores}")
    print(f"Columna de palabras clave: {columna_keywords}")

    return {
        "data": df,
        "authors_frequency": authors_frequency,
        "country_frequency": country_frequency,
        "keywords_frequency": keywords_frequency,
        "source_frequency": source_frequency,
        "country_cooc_matrix": country_matrix,
        "keywords_cooc_matrix": keywords_matrix,
    }


resultados = main()

Documentos leídos: 405
Columnas disponibles: ['Authors', 'Author full names', 'Author(s) ID', 'Title', 'Year', 'Source title', 'Volume', 'Issue', 'Art. No.', 'Page start', 'Page end', 'Cited by', 'DOI', 'Link', 'Affiliations', 'Authors with affiliations', 'Abstract', 'Author Keywords', 'Index Keywords', 'Abbreviated Source Title', 'Document Type', 'Publication Stage', 'Open Access', 'Source', 'EID']
Países inferidos del último componente de cada afiliación. Revisa country_frequency.csv para detectar nombres incorrectos.

Se generaron los 15 archivos.
Columna de autores: Authors
Columna de palabras clave: Author Keywords


In [2]:
%pip install plotly

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 4.8 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/9.7 MB 8.4 MB/s eta 0:00:01
   ---------------------------- ----------- 6.8/9.7 MB 11.7 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 13.1 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 12.1 MB/s  0:00:01

   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ---------------------------------------- 0/2 [narwhals]
   ------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
